# RecursiveCharacterTextSplitter

`RecursiveCharacterTextSplitter`는 **가장 권장되는 범용 텍스트 분할 방식**입니다.

## 주요 특징

- **재귀적 분할**: 여러 구분자를 우선순위에 따라 순차적으로 시도합니다.
- **계층적 접근**: 단락 → 문장 → 단어 순서로 분할하여 의미 단위를 보존합니다.
- **유연한 청크 크기**: 청크가 너무 크면 다음 구분자로 재귀적으로 재분할합니다.

## 동작 원리

1. **구분자 우선순위**: 기본값은 `["\n\n", "\n", " ", ""]`
   - `"\n\n"` (단락) → `"\n"` (줄) → `" "` (단어) → `""` (문자) 순서로 시도
2. **재귀적 분할**: 청크 크기가 `chunk_size`를 초과하면 다음 구분자로 재시도
3. **문맥 보존**: 가능한 한 큰 의미 단위를 유지하면서 분할

## CharacterTextSplitter와의 차이점

| 특징 | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|------|----------------------|-------------------------------|
| 구분자 | 단일 구분자만 사용 | 여러 구분자를 순차적으로 시도 |
| 분할 방식 | 단순 분할 | 재귀적 분할 |
| 청크 크기 관리 | 초과 시 그대로 유지 | 초과 시 자동 재분할 |
| 권장 용도 | 단순 문서 | **대부분의 경우 (권장)** |

## 사용 시나리오

- ✅ **일반 텍스트 문서**: 대부분의 경우 최선의 선택
- ✅ **복잡한 문서 구조**: 단락, 문장, 단어 혼합 문서
- ✅ **의미 보존 중요**: 문맥을 유지하면서 분할이 필요할 때


# 02. RecursiveCharacterTextSplitter

## 학습 목표
- `RecursiveCharacterTextSplitter`의 재귀적 분할 원리를 이해해요
- 기본 구분자 우선순위(`\n\n` → `\n` → ` ` → `""`)의 동작 방식을 파악해요
- `CharacterTextSplitter`와의 차이를 실제 데이터로 비교해요

## 사전 지식
- 01-CharacterTextSplitter 완료

---

> **이전 노트북에서 배운 것**: CharacterTextSplitter는 단일 구분자만 사용해요. 청크가 너무 커도 자동으로 재분할하지 않아요.
>
> **이번 노트북에서 배울 것**: 여러 구분자를 **우선순위 순서**로 시도해서 의미 단위를 최대한 보존하며 분할해요. **대부분의 프로젝트에서 이 방식을 기본값으로 사용해요.**


> 🔑 **핵심 개념**: `RecursiveCharacterTextSplitter`는 RAG 프로젝트의 **기본값**입니다. `\n\n` → `\n` → ` ` → `""` 순서로 구분자를 시도해서 청크가 `chunk_size`를 초과하면 자동으로 더 작은 단위로 재분할합니다.

## 샘플 데이터 로드

동일한 기술 용어 사전 데이터를 사용하여 RecursiveCharacterTextSplitter의 동작을 확인합니다.


In [1]:
# 샘플 데이터 파일을 읽어옵니다.
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()


로드된 텍스트의 일부를 확인합니다.


In [2]:
# 파일 내용 중 처음 500자를 출력합니다.
print(file[:500])
print(f"\n[전체 텍스트 길이: {len(file)} 문자]")


Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.
예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.
연관키워드: 토큰화, 자연어

[전체 텍스트 길이: 5733 문자]


## 1. 기본 사용법

`RecursiveCharacterTextSplitter`의 기본 설정으로 텍스트를 분할합니다.

### 주요 파라미터

- `chunk_size`: 청크의 최대 문자 수
- `chunk_overlap`: 청크 간 중복 영역
- `length_function`: 길이 계산 함수 (기본값: `len`)
- `is_separator_regex`: 구분자를 정규식으로 해석할지 여부 (기본값: `False`)
- `separators`: 분할에 사용할 구분자 리스트 (기본값: `["\n\n", "\n", " ", ""]`)


> 🎯 **강의 포인트**: `CharacterTextSplitter`와 `RecursiveCharacterTextSplitter`는 같은 파라미터를 가지지만 동작이 다릅니다. Recursive는 청크가 크면 더 작은 구분자로 재시도하므로 `chunk_size`를 더 잘 준수합니다.

In [3]:
# ---------------------------------------------------
# RecursiveCharacterTextSplitter 생성
# ---------------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1단계: RecursiveCharacterTextSplitter 생성
# - 기본 구분자 우선순위: ["\n\n", "\n", " ", ""]
#   단락 → 줄 → 단어 → 문자 순서로 시도하여 최적 분할점 탐색
# - chunk_size=250: 청크 최대 문자 수
# - chunk_overlap=50: 인접 청크 간 중복 (문맥 연결)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,  # 청크 최대 크기: 250자
    chunk_overlap=50,  # 청크 간 중복: 50자
    length_function=len,  # 길이 계산 함수
    is_separator_regex=False,  # 정규식 사용 안 함
)

## 2. 재귀적 분할 확인

텍스트를 분할하고 결과를 확인합니다. 첫 번째와 두 번째 청크를 비교하여 재귀적 분할이 어떻게 작동하는지 살펴봅니다.


In [4]:
# ============================================================
# TODO: RecursiveCharacterTextSplitter로 텍스트를 분할해보세요
# 힌트: text_splitter.create_documents([file]) → Document 리스트 반환
# 예상 결과: 30개의 Document가 생성됩니다
# ============================================================

# 1단계: 텍스트를 Document 객체 리스트로 분할
documents = text_splitter.create_documents([file])

print(f"생성된 문서 개수: {len(documents)}")
print(f"첫 번째 청크 길이: {len(documents[0].page_content)} 문자")
print("\n" + "=" * 60)
print("📄 첫 번째 청크:")
print("=" * 60)
print(documents[0].page_content)

생성된 문서 개수: 30
첫 번째 청크 길이: 197 문자

📄 첫 번째 청크:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding


두 번째 청크를 확인하여 청크 간 중복(`chunk_overlap`)이 어떻게 작동하는지 살펴봅니다.


In [5]:
# 두 번째 청크 확인
print("=" * 60)
print("📄 두 번째 청크:")
print("=" * 60)
print(documents[1].page_content)


📄 두 번째 청크:
Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token


## 3. split_text() 메서드 사용

`split_text()` 메서드를 사용하면 순수 문자열 리스트를 반환받을 수 있습니다.


In [6]:
# 텍스트를 문자열 리스트로 분할
text_chunks = text_splitter.split_text(file)

print(f"분할된 텍스트 청크 개수: {len(text_chunks)}")
print("\n처음 2개 청크를 확인합니다:")
print("\n" + "=" * 60)
print("📝 첫 번째 텍스트 청크:")
print("=" * 60)
print(text_chunks[0])
print("\n" + "=" * 60)
print("📝 두 번째 텍스트 청크:")
print("=" * 60)
print(text_chunks[1])


분할된 텍스트 청크 개수: 30

처음 2개 청크를 확인합니다:

📝 첫 번째 텍스트 청크:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

📝 두 번째 텍스트 청크:
Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.
연관키워드: 자연어 처리, 벡터화, 딥러닝

Token


## 4. CharacterTextSplitter와 비교

동일한 파라미터로 `CharacterTextSplitter`와 `RecursiveCharacterTextSplitter`의 결과를 비교합니다.


In [7]:
# ============================================================
# TODO: CharacterTextSplitter와 RecursiveCharacterTextSplitter의 분할 결과를 비교해보세요
# 힌트: 동일한 파라미터로 두 Splitter를 생성하고 split_text()로 분할한 후 개수와 평균 크기를 비교
# 예상 결과: 두 방식의 청크 개수와 평균 크기가 출력됩니다
# ============================================================

from langchain_text_splitters import CharacterTextSplitter

# 1단계: CharacterTextSplitter로 분할
char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
)
char_chunks = char_splitter.split_text(file)

# 2단계: RecursiveCharacterTextSplitter로 분할
recursive_chunks = text_splitter.split_text(file)

print("=" * 60)
print("📊 분할 결과 비교")
print("=" * 60)
print(f"CharacterTextSplitter 청크 개수: {len(char_chunks)}")
print(f"RecursiveCharacterTextSplitter 청크 개수: {len(recursive_chunks)}")
print(f"\n청크 크기 분석:")
print(f"  CharacterTextSplitter 평균 크기: {sum(len(c) for c in char_chunks) / len(char_chunks):.1f} 문자")
print(f"  RecursiveCharacterTextSplitter 평균 크기: {sum(len(c) for c in recursive_chunks) / len(recursive_chunks):.1f} 문자")

📊 분할 결과 비교
CharacterTextSplitter 청크 개수: 30
RecursiveCharacterTextSplitter 청크 개수: 30

청크 크기 분석:
  CharacterTextSplitter 평균 크기: 203.5 문자
  RecursiveCharacterTextSplitter 평균 크기: 201.8 문자


> 💡 **실무 팁**: 한글 문서에서는 `separators`에 `". "`, `"? "`, `"! "`를 추가하면 문장 경계를 더 잘 인식합니다. 영문 기준 `chunk_size=1000`이라면 한글은 `chunk_size=800`으로 줄이는 것이 좋습니다.

> ⚠️ **자주 하는 실수**: `is_separator_regex=False` (기본값)일 때 `\n`은 리터럴 문자열로 처리됩니다. 정규식 패턴을 구분자로 사용하려면 반드시 `is_separator_regex=True`를 설정하세요.

## 5. 사용자 정의 구분자

기본 구분자 외에 한글 문서에 적합한 구분자를 추가할 수 있습니다.


In [8]:
# ============================================================
# TODO: 한글 문장 부호를 포함한 사용자 정의 separators를 RecursiveCharacterTextSplitter에 추가해보세요
# 힌트: separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""]
# 예상 결과: 기본 구분자와 청크 개수 비교 결과가 출력됩니다
# ============================================================

# 1단계: 한글 문서에 적합한 구분자 정의
# - 한글 문장 부호(". ", "? ", "! ")를 추가해서 문장 단위 분할 가능
korean_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    separators=[
        "\n\n",  # 단락 구분
        "\n",    # 줄 구분
        ". ",    # 마침표 + 공백
        "? ",    # 물음표 + 공백
        "! ",    # 느낌표 + 공백
        " ",     # 공백
        "",      # 문자
    ],
    length_function=len,
)

# 2단계: 분할 실행
korean_chunks = korean_text_splitter.split_text(file)

print(f"한글 최적화 구분자 사용 청크 개수: {len(korean_chunks)}")
print(f"기본 구분자 사용 청크 개수: {len(recursive_chunks)}")
print(f"\n첫 번째 청크 비교:")
print("=" * 60)
print(korean_chunks[0])

한글 최적화 구분자 사용 청크 개수: 30
기본 구분자 사용 청크 개수: 30

첫 번째 청크 비교:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding


## 6. 청크 크기 실험

다양한 `chunk_size` 값을 테스트하여 적절한 크기를 찾아봅니다.


## 핵심 정리 및 마무리

### 문서 유형별 권장 chunk_size

| 문서 유형 | chunk_size | chunk_overlap |
|----------|:---------:|:-------------:|
| 짧은 Q&A, 뉴스 | 500 | 50 |
| 일반 문서, 블로그 | 1000 | 100 |
| 기술 문서, 논문 | 1500 | 150 |
| 책, 매뉴얼 | 2000 | 200 |

### 한글 문서 팁
> 한글은 영어보다 토큰 수가 많아요. 영문 기준 `chunk_size=1000`이라면 한글 문서는 `chunk_size=800` 정도로 조정하는 게 좋아요. 마침표나 물음표를 구분자에 추가하면 더 자연스럽게 분할돼요.

---

## 다음 예고

문자(character) 기반 분할에서 **토큰(token)** 기반 분할로 넘어가요.

- **03-TokenTextSplitter**: LLM 토큰 제한을 정확히 준수하는 분할 방식
- **04-SemanticChunker**: 임베딩 유사도로 의미 단위를 찾는 고급 분할
